# **Assignment - 5 Emerging Models Research**

**Model Demonstrated:** Qwen2.5-VL-7B-Instruct  

**GitHub Link**: https://github.com/Venkata-Murari1711/Generative-AI-model-development.git

## **Environment Setup**

In [ ]:
# Install required libraries for running the Qwen2.5-VL-7B model
!pip install -q accelerate pillow # accelerate: helps Hugging Face efficiently manage GPU usage and device mapping
!pip install -q git+https://github.com/huggingface/transformers # Install the latest development version of Transformers from GitHub
!pip install -q qwen-vl-utils # For video/multimodal support

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## **Importing Libraries**

In [ ]:
import torch

# Import Qwen2.5-VL model and processor to load and prepare inputs for the vision-language model
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

from PIL import Image # Import PIL for loading and handling image files
import matplotlib.pyplot as plt
import json # Import json for formatting structured data outputs if needed
import requests
from io import BytesIO # Import BytesIO to convert downloaded image data into a readable file format

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## **Loading Model**

In this step, the Qwen2.5-VL-7B-Instruct vision-language model is loaded from Hugging Face.
The model is configured to use **bfloat16 precision** for efficient GPU computation in Google Colab.
Automatic device mapping allows the model to utilize the available GPU for faster inference.
After loading, the processor is initialized to prepare image and text inputs for the model.
Finally, the environment verifies the GPU being used and reports the memory currently allocated.

In [ ]:
# Load the Qwen2.5-VL-7B-Instruct vision-language model from Hugging Face
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.bfloat16, # Use bfloat16 precision to reduce GPU memory usage
    device_map="auto", # Automatically assign the model to the available GPU
    trust_remote_code=True # Allow custom model code provided by the model repository
)

# Load the processor that prepares image and text inputs for the model
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", trust_remote_code=True)

print("Model loaded successfully!")

# Check if a GPU is available for computation
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading Qwen2.5-VL-7B-Instruct... This takes 3-5 minutes.
Using bfloat16 precision (no quantization required)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded successfully!
GPU: Tesla T4
GPU Memory used: 12.70 GB


## **Vision-Language Inference Function**

This function runs the **Qwen2.5-VL-7B vision-language model** on an image and a text prompt.  
It prepares the input in the correct format, processes the visual information, sends it to the model for inference, and returns the generated response.  

The function is reused throughout the notebook to test different prompts for **Image-Based Question Answering** and **Chart/Graph Analysis**.

In [ ]:
# Import utility function used to process image/video inputs for Qwen VL models
from qwen_vl_utils import process_vision_info

def run_qwen_vl(image_path, prompt, max_new_tokens=200):
    # A Message structure combining the image input and text prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{image_path}"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Convert the message into the chat template format required by the model
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Extract image/video inputs for the model using Qwen vision utilities
    image_inputs, video_inputs = process_vision_info(messages)

    # Prepare the final inputs for the model including text and image tensors
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate a response from the model based on the inputs
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode the generated token IDs into readable text
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Return the model's generated response
    return output_text

## **Case 1: Image-Based Question Answering**
In this example, Qwen2.5-VL-7B is used to analyze a natural scene image.
The model is prompted to describe the image, identify visible elements,
and infer the possible purpose of the image.


### **Upload Image**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Demo image.jpg to Demo image.jpg


### **Prompt 1**

In [ ]:
image_path = list(uploaded.keys())[0]

prompt = """Answer using only what you can see in the image.
1. What is this image about?
2. List 5 important visible details.
3. What is one likely purpose of this image?
4. Mention anything unclear or unreadable."""
print(run_qwen_vl(image_path, prompt))

system
You are a helpful assistant.
user
Answer using only what you can see in the image.
1. What is this image about?
2. List 5 important visible details.
3. What is one likely purpose of this image?
4. Mention anything unclear or unreadable.
assistant
1. This image depicts a group of people jumping joyfully on a beach at sunset.

2. Important visible details:
   - Four silhouetted figures in mid-air, arms raised.
   - The beach setting with sand and water.
   - A sunset sky with warm hues transitioning to cooler tones.
   - The figures appear to be in a celebratory pose.
   - The horizon line where the ocean meets the sky.

3. One likely purpose of this image could be to convey a sense of freedom, happiness, or celebration.

4. There is no text or unclear elements in the image that need to be addressed.


### **Analysis**

The model successfully identified the main elements of the image,
including the number of people, their activity, and the sunset beach environment.
It also correctly inferred the emotional tone of the scene (celebration and happiness).

However, the model makes a small assumption about the purpose of the image,
which demonstrates how vision-language models may combine visual cues with
general world knowledge when generating responses.

### **Prompt 2**

In [ ]:
prompt = """Extract the important visible information from this image.
Return:
- main title or heading
- important labels
- numbers or dates
- short summary"""
print(run_qwen_vl(image_path, prompt))

system
You are a helpful assistant.
user
Extract the important visible information from this image.
Return:
- main title or heading
- important labels
- numbers or dates
- short summary
assistant
- **Main Title/Heading**: Joyful Sunset Celebration
- **Important Labels**: None
- **Numbers or Dates**: None
- **Short Summary**: A group of four people is joyfully jumping on a beach at sunset, creating silhouettes against the colorful sky and ocean. The scene conveys a sense of happiness and freedom.


### **Analysis**

The model correctly recognized that there are no visible titles, labels,
or numerical data in the image. It summarized the scene accurately,
showing strong visual understanding and contextual interpretation.

## **Case 2: Chart and Graph Understanding**

In this case, the model analyzes a chart image and attempts to interpret
data trends, identify key points, and summarize insights.
This demonstrates the model's capability to understand visual data
representations such as graphs and charts.

### **Upload Chart**

In [ ]:
from google.colab import files
uploaded_file2 = files.upload()

Saving Chart Demo.png to Chart Demo.png


### **Prompt 1**

In [ ]:
chart_path = list(uploaded_file2.keys())[0]

prompt = """Analyze this chart.
1. What type of chart is it?
2. What are the main variables or categories shown?
3. Summarize the overall trend.
4. Identify the highest and lowest points approximately.
5. Mention any missing information that limits confidence."""
print(run_qwen_vl(chart_path, prompt, max_new_tokens=250))

system
You are a helpful assistant.
user
Analyze this chart.
1. What type of chart is it?
2. What are the main variables or categories shown?
3. Summarize the overall trend.
4. Identify the highest and lowest points approximately.
5. Mention any missing information that limits confidence.
assistant
1. The chart is a combination of a bar chart and a line chart, showing two different types of data over time.

2. The main variables or categories shown are:
   - "Units Sold" (represented by blue bars)
   - "Total Transaction" (represented by an orange line)

3. The overall trend shows an increase in both units sold and total transactions from January to June. There is a dip in March for both metrics, but they recover and continue to rise in April and May before reaching their peak in June.

4. The approximate highest point for "Units Sold" is around 150 units in June, and the highest point for "Total Transaction" is around 7500 in June. The lowest points appear to be around 30 units sold a

### **Analysis**

The model correctly identified the chart type and the variables represented
by the bars and line. It also summarized the overall trend accurately,
highlighting the increase from January to June and the dip in March.

The model demonstrates strong reasoning abilities by estimating approximate
values and identifying missing information such as axis scales or exact data values.

### **Prompt 2**

In [ ]:
prompt = """Read this chart and provide:
- a 3-bullet summary
- the most important trend
- one practical conclusion someone could make from this chart"""
print(run_qwen_vl(chart_path, prompt, max_new_tokens=220))

system
You are a helpful assistant.
user
Read this chart and provide:
- a 3-bullet summary
- the most important trend
- one practical conclusion someone could make from this chart
assistant
### Summary:
1. **Units Sold**: The number of units sold shows a general upward trend, with significant increases in February and June.
2. **Total Transactions**: The total transactions also show an increasing trend but with more fluctuations compared to units sold.
3. **March Dip**: There is a noticeable dip in both units sold and total transactions in March.

### Most Important Trend:
The most important trend is the overall increase in both units sold and total transactions over the period from January to June, with a notable peak in June for both metrics.

### Practical Conclusion:
A practical conclusion someone could make from this chart is that there is a strong correlation between increased sales volume and higher transaction amounts, suggesting that strategies implemented in February and June

### **Analysis**

The model generated a clear summary of the chart's trends and relationships
between units sold and total transactions. It also provided a practical
business interpretation of the data, which demonstrates the usefulness of
vision-language models for decision support and analytics.

## **Overall Observations and Limitations**

The experiments demonstrate that Qwen2.5-VL-7B is capable of interpreting
both natural images and structured visual data such as charts.

Strengths observed in the experiments include:
- Accurate description of visual scenes
- Ability to extract meaningful insights from charts
- Strong contextual reasoning about visual content

However, several limitations were also observed:
- Exact numerical extraction from charts can be uncertain
- The model may make assumptions about image purpose
- Performance depends heavily on image clarity and resolution

Overall, the results show that vision-language models like Qwen2.5-VL-7B
can be valuable tools for visual understanding, data interpretation,
and interactive AI assistants.